# LoRA-ESM2 Hybrid v1 — development-only screen

This notebook tests a faster ESM-2 hybrid with three complementary experts: LoRA-adapted contextual ESM-2 features, a multi-scale local-motif CNN, and a bidirectional-AAindex branch. It trains seed 42 for all 15 epochs on the same protein-grouped development split used by CenterLoRA-ESM2 v1. It does **not** load the released MMUbiPred test set or perform a full-data refit. Select a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Add a fine-grained read-only GitHub token to Colab Secrets as `GITHUB_TOKEN`. Optionally add `HF_TOKEN` to avoid unauthenticated Hugging Face rate limits. Enable notebook access for each secret.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
except ImportError:
    GITHUB_TOKEN = None
    HF_TOKEN = None

git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
askpass_path = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    git_environment['GIT_ASKPASS'] = str(askpass_path)
    git_environment['GITHUB_TOKEN'] = GITHUB_TOKEN

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--recurse-submodules',
            'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
            str(REPO_ROOT),
        ], check=True, env=git_environment)
    else:
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
            check=True, env=git_environment,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
            check=True, env=git_environment,
        )
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Git access failed. Add GITHUB_TOKEN to Colab Secrets for a private repository.'
    ) from error
finally:
    askpass_path.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = str(
    Path('/content/drive/MyDrive/MMUbiPred-research/huggingface-cache')
)
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
print('Project:', REPO_ROOT)
print('Persistent Hugging Face cache:', os.environ['HF_HOME'])
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import json
import platform
import sys
import peft
import sklearn
import torch
import transformers

print('Python:', sys.version)
print('Platform:', platform.platform())
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Select a GPU runtime before starting the screen.')

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError('Could not open subprocess output stream')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Preflight tests

In [ ]:
run_live([
    sys.executable, '-u', '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
])

## Run the predeclared seed-42 screen

The Drive-backed run is restart-safe. If Colab disconnects after an epoch, rerunning this cell resumes at the next epoch. All 15 epochs run, while the best development checkpoint at threshold 0.5 is retained.

In [ ]:
RUN_DIR = Path(
    '/content/drive/MyDrive/MMUbiPred-research/experiments/'
    'lora-esm2-hybrid-v1-seed42'
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RUN_DIR / 'validation_metrics.json'
RESUME_PATH = RUN_DIR / 'development_resume.pt'

if METRICS_PATH.exists():
    print('Completed metrics already exist; training was skipped:', METRICS_PATH)
else:
    if RESUME_PATH.exists():
        resume_state = torch.load(RESUME_PATH, map_location='cpu', weights_only=False)
        print('Resuming after completed epoch:', resume_state['completed_epoch'])
        del resume_state
    run_live([
        sys.executable, '-u', 'experiment/scripts/train.py',
        '--config', 'experiment/configs/lora_esm2_hybrid_screen_v1.json',
        '--seed', '42',
        '--output-dir', RUN_DIR,
        '--development-only',
        '--resume-development',
    ])
print('Persistent run directory:', RUN_DIR)

## Compare with the frozen ESM-2 v1 seed-42 reference

The notebook locates a complete 15-epoch ESM-2 v1 result and rejects the comparison unless both models used the identical validation-index hash.

In [ ]:
V1_METRICS_PATH = Path(
    '/content/drive/MyDrive/MMUbiPred-research/benchmarks/'
    'center-lora-esm2-full15-three-seed-v1/'
    'center_lora_esm2_full15_v1/seed_42/validation_metrics.json'
)

def valid_full15_v1_metrics(path):
    try:
        metrics = json.loads(path.read_text())
        manifest = json.loads((path.parent / 'run_manifest.json').read_text())
        config = manifest['config']
        return (
            int(config['seed']) == 42
            and int(config['epochs']) == 15
            and config['experiment_name'] == 'center_lora_esm2_full15_v1'
            and metrics['development_selection']['epochs_completed'] == 15
            and not manifest['locked_test_accessed']
        )
    except Exception:
        return False

if not valid_full15_v1_metrics(V1_METRICS_PATH):
    archive_root = Path('/content/drive/MyDrive/MMUbiPred-research')
    matches = [
        path for path in archive_root.rglob('validation_metrics.json')
        if path != METRICS_PATH and valid_full15_v1_metrics(path)
    ]
    if not matches:
        raise FileNotFoundError(
            'No complete full-15 seed-42 ESM-2 v1 result was found on Drive.'
        )
    V1_METRICS_PATH = max(matches, key=lambda path: path.stat().st_mtime)

v1_metrics = json.loads(V1_METRICS_PATH.read_text())
hybrid_metrics = json.loads(METRICS_PATH.read_text())
v1_manifest = json.loads((V1_METRICS_PATH.parent / 'run_manifest.json').read_text())
hybrid_manifest = json.loads((METRICS_PATH.parent / 'run_manifest.json').read_text())

v1_hash = v1_manifest['development_split']['validation_indices_sha256']
hybrid_hash = hybrid_manifest['development_split']['validation_indices_sha256']
if v1_hash != hybrid_hash:
    raise RuntimeError(
        f'Validation split mismatch: v1={v1_hash}, hybrid={hybrid_hash}'
    )

diagnostic_key = 'mean_' + hybrid_metrics['branch_diagnostic_name']
display({'v1_metrics_path': str(V1_METRICS_PATH), 'validation_hash': v1_hash})
display({
    'hybrid_fixed_threshold': hybrid_metrics['fixed_threshold'],
    'hybrid_selected_threshold': hybrid_metrics['validation_selected_threshold'],
    'hybrid_development_selection': hybrid_metrics['development_selection'],
    'hybrid_branch_names': hybrid_metrics['branch_names'],
    diagnostic_key: hybrid_metrics[diagnostic_key],
    'hybrid_prediction_diagnostics': hybrid_metrics['prediction_diagnostics'],
})

In [ ]:
metric_keys = ['mcc', 'accuracy', 'sensitivity', 'specificity', 'auroc', 'auprc']
comparison = {
    metric: {
        'esm2_v1': float(v1_metrics['fixed_threshold'][metric]),
        'hybrid_v1': float(hybrid_metrics['fixed_threshold'][metric]),
        'hybrid_minus_esm2': (
            float(hybrid_metrics['fixed_threshold'][metric])
            - float(v1_metrics['fixed_threshold'][metric])
        ),
    }
    for metric in metric_keys
}
comparison['selected_mcc'] = {
    'esm2_v1': float(v1_metrics['validation_selected_threshold']['mcc']),
    'hybrid_v1': float(hybrid_metrics['validation_selected_threshold']['mcc']),
    'hybrid_minus_esm2': (
        float(hybrid_metrics['validation_selected_threshold']['mcc'])
        - float(v1_metrics['validation_selected_threshold']['mcc'])
    ),
}
display(comparison)

PROTBERT_FIXED_MCC = 0.5827515027503104
mcc_gain = comparison['mcc']['hybrid_minus_esm2']
selected_safe = comparison['selected_mcc']['hybrid_v1'] >= 0.580313826500746
ranking_safe = (
    comparison['auroc']['hybrid_minus_esm2'] >= -0.002
    and comparison['auprc']['hybrid_minus_esm2'] >= -0.002
)
collapsed = bool(
    hybrid_metrics['prediction_diagnostics']['single_class_predictions_at_0_5']
)
safety_conditions = selected_safe and ranking_safe and not collapsed

if safety_conditions and comparison['mcc']['hybrid_v1'] >= PROTBERT_FIXED_MCC:
    decision = (
        'GO: the hybrid matched or exceeded the ProtBERT development champion; '
        'report before any multi-seed confirmation.'
    )
elif safety_conditions and mcc_gain >= 0.010:
    decision = (
        'PROMISING BUT NOT CHAMPION: material ESM-2 gain, but below ProtBERT; '
        'do not promote or run more seeds yet.'
    )
else:
    decision = 'STOP HYBRID V1: it did not satisfy the predeclared screen.'

print(decision)
print('Released MMUbiPred test loaded by this notebook:', False)
print('A future final claim requires a newly curated blind external test set.')

## What to report

Send the two displayed dictionaries and the decision line. Do not add a released-test evaluation cell: that dataset has already served as a historical benchmark for an earlier selected candidate and must not guide this architecture.